In [ ]:
# Célula 0 — Tokens, Drive e código do repositório
import os
from google.colab import drive, userdata

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # opcional, acelera downloads
except Exception:
    print("HF_TOKEN não configurado (ok, só fica mais lento).")

drive.mount("/content/drive")

REPO = "/content/experimentos-llm-local"
if not os.path.exists(REPO):
    !git clone https://github.com/Arthur-galvao/experimentos-llm-local.git {REPO}
else:
    !git -C {REPO} pull
%cd {REPO}

# F0b — Teste de equivalência: vLLM oficial (PR #60149) vs `lib/dory.py`

**Critério:** 5 prompts × loops {1, 2, 4} × 32 tokens gulosos. O vLLM gera; a nossa reimplementação recebe exatamente a mesma sequência (teacher-forcing) e comparamos o top-1 posição a posição. Divergência > 2% = reprovado.

**Como este notebook evita os erros anteriores:**
- O cache do Hugging Face fica **sempre no disco local** (`/root/.cache/huggingface`), nunca no Drive. O Drive corrompia o `tokenizer.json`.
- Os arquivos do modelo são validados (JSON íntegro) antes de qualquer uso.
- Cada número de loops roda num **processo Python separado**, o que libera 100% da GPU entre um e outro. Não é preciso reiniciar a sessão.

In [ ]:
# Célula 1 — Instalação (o vLLM da PR traz transformers 5.x e afins; não fixar versões por cima)
!VLLM_USE_PRECOMPILED=1 pip install -q "git+https://github.com/vllm-project/vllm.git@refs/pull/60149/head"
!pip install -q accelerate
!python -c "import transformers, vllm; print('transformers', transformers.__version__, '| vllm', vllm.__version__)"

In [ ]:
# Célula 2 — Cache do HF no disco local + validação dos arquivos do modelo
import os, json, shutil
from pathlib import Path

LOCAL_HF = "/root/.cache/huggingface"
os.environ["HF_HOME"] = LOCAL_HF
os.environ["HF_HUB_CACHE"] = f"{LOCAL_HF}/hub"
os.environ["HF_DATASETS_CACHE"] = f"{LOCAL_HF}/datasets"

# Remove a cópia corrompida antiga no Drive (se existir)
shutil.rmtree("/content/drive/MyDrive/coder-comprimido/hf_cache", ignore_errors=True)

from huggingface_hub import hf_hub_download, snapshot_download

DORY = "nvidia/NVIDIA-Nemotron-Labs-Dory1.0-4B-BF16"
snapshot_download(DORY)  # baixa o que faltar (pesos ~8GB na primeira vez)

for fname in ("config.json", "tokenizer_config.json", "tokenizer.json"):
    p = hf_hub_download(DORY, fname)
    try:
        json.load(open(p))
        print(f"[OK] {fname}")
    except Exception:
        print(f"[CORROMPIDO] {fname} -> baixando de novo")
        p = hf_hub_download(DORY, fname, force_download=True)
        json.load(open(p))
        print(f"[OK após novo download] {fname}")

RESULTS_DIR = Path("/content/drive/MyDrive/coder-comprimido/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Resultados em:", RESULTS_DIR)

In [ ]:
%%writefile /content/run_equiv.py
# Célula 3 — Script que roda UM número de loops (executado em processo separado).
# Todo o código fica dentro de main() com a guarda `if __name__ == "__main__"`: o vLLM cria
# o motor com multiprocessing "spawn", que reimporta este arquivo; sem a guarda, o teste
# rodaria de novo dentro do processo filho e o motor quebraria na inicialização.
import os, sys

os.environ["HF_HOME"] = "/root/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/root/.cache/huggingface/hub"
os.environ["HF_DATASETS_CACHE"] = "/root/.cache/huggingface/datasets"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
sys.path.insert(0, "/content/experimentos-llm-local")


def main() -> None:
    # vLLM usa no máximo 60% da VRAM e contexto de 2048 tokens (>= sliding window de 1024).
    # Sobra espaço; o motor roda em processo próprio e libera a GPU antes do modelo HF carregar.
    import vllm

    _OrigLLM = vllm.LLM

    class _LLMReduzido(_OrigLLM):
        def __init__(self, *args, **kwargs):
            kwargs["gpu_memory_utilization"] = 0.60
            kwargs["max_model_len"] = 2048
            super().__init__(*args, **kwargs)

    vllm.LLM = _LLMReduzido

    from lib.dory import DEFAULT_EQUIVALENCE_PROMPTS, DORY_HF_REPO, verify_vllm_hf_equivalence

    k = int(sys.argv[1])
    out = f"/content/drive/MyDrive/coder-comprimido/results/equivalence_vllm_hf_L{k}.json"
    try:
        verify_vllm_hf_equivalence(
            prompts=DEFAULT_EQUIVALENCE_PROMPTS,
            loops=(k,),
            max_new_tokens=32,
            max_divergence_rate=0.02,
            model_name_or_path=DORY_HF_REPO,
            output_json_path=out,
            halt_on_failure=True,
        )
        print(f"\n=== LOOPS={k}: APROVADO === ({out})")
    except RuntimeError as e:
        msg = str(e)
        if "diverg" in msg.lower():
            print(f"\n=== LOOPS={k}: REPROVADO ===\n{msg}")
            sys.exit(2)
        raise  # erro de execução (ex.: vLLM não abriu) -> não é reprovação


if __name__ == "__main__":
    main()

In [ ]:
# Célula 4 — Roda loops 1, 2 e 4, cada um num processo novo (GPU liberada entre eles)
import subprocess

status = {}
for k in (1, 2, 4):
    print(f"\n{'='*20} LOOPS = {k} {'='*20}")
    !nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader
    r = subprocess.run(["python", "/content/run_equiv.py", str(k)], capture_output=True, text=True)
    linhas = (r.stdout + r.stderr).strip().splitlines()
    if r.returncode not in (0, 2):
        # Em caso de erro, mostra primeiro as linhas com a causa real (ficam acima do traceback final)
        erros = [l for l in linhas if "Error" in l or "ERROR" in l or "error:" in l]
        print("--- linhas de erro ---")
        print("\n".join(erros[-25:]))
        idx = next((i for i, l in enumerate(linhas) if "Traceback" in l), max(0, len(linhas) - 80))
        print("--- início do primeiro traceback ---")
        print("\n".join(linhas[max(0, idx - 5): idx + 40]))
        print("--- final do log ---")
    print("\n".join(linhas[-40:]))
    status[k] = {0: "APROVADO", 2: "REPROVADO"}.get(r.returncode, f"ERRO (código {r.returncode})")

print("\nRESUMO:", status)